# **Semantic Chunking**

1. SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

2. It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [28]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# load environment variables
load_dotenv()

True

In [27]:
# sample text
text = """
Langchain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents,  memory and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

# step 1: Split into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# step 2: Embed each sentences
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("IQ_BASE_URL"),
    model=os.getenv("EMBEDDING_MODEL"),
    dimensions=1536,
)

embedded_sentences = [embeddings.embed_query(sentence) for sentence in sentences]

THRESHOLD = 0.7
chunks = []
current_chunk = [sentences[0]]
for idx in range(1, len(sentences)):
    sim = cosine_similarity(
        [embedded_sentences[idx-1]],
        [embedded_sentences[idx]],
    )[0][0]

    if sim >= THRESHOLD:
        current_chunk.append(sentences[idx])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[idx]]

chunks.append(" ".join(current_chunk))

chunks

['Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents,  memory and retrievers.',
 'The Eiffel Tower is located in Paris.',
 'France is a popular tourist destination.']